In [1]:
from pyspark.sql import SparkSession
import getpass
username =getpass.getuser()
spark =SparkSession.builder.config('spark.ui.port','0') \
.config('spark.shuffle.useOldFetchProtocol','true') \
.config('spark.sql.warehouse.dir', f'/user/{username}/warehouse') \
.enableHiveSupport() \
.master('yarn') \
.getOrCreate()

In [2]:
from pyspark.sql.functions import current_timestamp,regexp_replace,col,when,count

In [3]:
loans_schema ='loan_id string,member_id string,loan_amount float, funded_amount float,loan_term_months string,interest_rate float,monthly_installment float,issue_date string,loan_status string, loan_purpose string,loan_title string'

In [4]:
loans_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_schema) \
.load(f"/user/{username}/lendingclub/raw/loans_data_csv")

In [5]:
loans_df

loan_id,member_id,loan_amount,funded_amount,loan_term_months,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,loan_title
76003861,a4ec00ba67fadf2fe...,24000.0,24000.0,60 months,15.31,574.88,Apr-2016,Charged Off,debt_consolidation,Debt consolidation
76263914,4f7a9e6ffaacd5da2...,2400.0,2400.0,36 months,11.47,79.11,Apr-2016,Fully Paid,debt_consolidation,Debt consolidation
75537401,e935a4c27fc78ae61...,12600.0,12600.0,36 months,7.39,391.31,Apr-2016,Fully Paid,other,Other
75038986,2d32004bd5e1dc3c3...,16800.0,16800.0,60 months,19.53,440.72,Apr-2016,Current,credit_card,Credit card refin...
76301424,f7116b7f7546a7952...,4300.0,4300.0,36 months,17.27,153.89,Apr-2016,Charged Off,debt_consolidation,Debt consolidation
75333198,d3aa3a3c95eca5631...,8950.0,8950.0,36 months,22.45,343.9,Apr-2016,Current,credit_card,Credit card refin...
76391453,fc8a2e046eaaba02d...,35000.0,35000.0,60 months,12.99,796.18,Apr-2016,Fully Paid,debt_consolidation,Debt consolidation
76363364,577ae670ac2ec7ed3...,15000.0,15000.0,36 months,9.16,478.12,Apr-2016,Fully Paid,house,Home buying
76272510,d3792868819649ba9...,30000.0,30000.0,60 months,16.29,734.18,Apr-2016,Current,debt_consolidation,Debt consolidation
76304116,6d3a5a422261348b3...,4800.0,4800.0,36 months,19.99,178.37,Apr-2016,Fully Paid,credit_card,Credit card refin...


In [6]:
loans_df_ingested = loans_df.withColumn("ingest_date",current_timestamp())

In [7]:
loans_df_ingested

loan_id,member_id,loan_amount,funded_amount,loan_term_months,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,loan_title,ingest_date
76003861,a4ec00ba67fadf2fe...,24000.0,24000.0,60 months,15.31,574.88,Apr-2016,Charged Off,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
76263914,4f7a9e6ffaacd5da2...,2400.0,2400.0,36 months,11.47,79.11,Apr-2016,Fully Paid,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
75537401,e935a4c27fc78ae61...,12600.0,12600.0,36 months,7.39,391.31,Apr-2016,Fully Paid,other,Other,2026-03-28 07:19:...
75038986,2d32004bd5e1dc3c3...,16800.0,16800.0,60 months,19.53,440.72,Apr-2016,Current,credit_card,Credit card refin...,2026-03-28 07:19:...
76301424,f7116b7f7546a7952...,4300.0,4300.0,36 months,17.27,153.89,Apr-2016,Charged Off,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
75333198,d3aa3a3c95eca5631...,8950.0,8950.0,36 months,22.45,343.9,Apr-2016,Current,credit_card,Credit card refin...,2026-03-28 07:19:...
76391453,fc8a2e046eaaba02d...,35000.0,35000.0,60 months,12.99,796.18,Apr-2016,Fully Paid,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
76363364,577ae670ac2ec7ed3...,15000.0,15000.0,36 months,9.16,478.12,Apr-2016,Fully Paid,house,Home buying,2026-03-28 07:19:...
76272510,d3792868819649ba9...,30000.0,30000.0,60 months,16.29,734.18,Apr-2016,Current,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
76304116,6d3a5a422261348b3...,4800.0,4800.0,36 months,19.99,178.37,Apr-2016,Fully Paid,credit_card,Credit card refin...,2026-03-28 07:19:...


In [8]:
loans_df_ingested.createOrReplaceTempView("loans")

In [9]:
spark.sql("select count(*) from loans")

count(1)
2260701


In [10]:
spark.sql("select count(*) from loans where loan_amount is null")

count(1)
33


In [11]:
columns_to_check=["loan_amount","funded_amount","loan_term_months","interest_rate","monthly_installment","issue_date","loan_status","loan_purpose"]

In [12]:
# Drop where any of the cols is null,for checking all null colls,use ,"ANY" at the end
loans_filtered_df = loans_df_ingested.na.drop(subset=columns_to_check)

In [13]:
loans_filtered_df.count()

2260667

In [14]:
loans_filtered_df

loan_id,member_id,loan_amount,funded_amount,loan_term_months,interest_rate,monthly_installment,issue_date,loan_status,loan_purpose,loan_title,ingest_date
76003861,a4ec00ba67fadf2fe...,24000.0,24000.0,60 months,15.31,574.88,Apr-2016,Charged Off,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
76263914,4f7a9e6ffaacd5da2...,2400.0,2400.0,36 months,11.47,79.11,Apr-2016,Fully Paid,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
75537401,e935a4c27fc78ae61...,12600.0,12600.0,36 months,7.39,391.31,Apr-2016,Fully Paid,other,Other,2026-03-28 07:19:...
75038986,2d32004bd5e1dc3c3...,16800.0,16800.0,60 months,19.53,440.72,Apr-2016,Current,credit_card,Credit card refin...,2026-03-28 07:19:...
76301424,f7116b7f7546a7952...,4300.0,4300.0,36 months,17.27,153.89,Apr-2016,Charged Off,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
75333198,d3aa3a3c95eca5631...,8950.0,8950.0,36 months,22.45,343.9,Apr-2016,Current,credit_card,Credit card refin...,2026-03-28 07:19:...
76391453,fc8a2e046eaaba02d...,35000.0,35000.0,60 months,12.99,796.18,Apr-2016,Fully Paid,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
76363364,577ae670ac2ec7ed3...,15000.0,15000.0,36 months,9.16,478.12,Apr-2016,Fully Paid,house,Home buying,2026-03-28 07:19:...
76272510,d3792868819649ba9...,30000.0,30000.0,60 months,16.29,734.18,Apr-2016,Current,debt_consolidation,Debt consolidation,2026-03-28 07:19:...
76304116,6d3a5a422261348b3...,4800.0,4800.0,36 months,19.99,178.37,Apr-2016,Fully Paid,credit_card,Credit card refin...,2026-03-28 07:19:...


In [15]:
loans_filtered_df.createOrReplaceTempView("loans")

In [16]:
loans_term_modfied_df = loans_filtered_df.withColumn("loan_term_months",(regexp_replace(col("loan_term_months")," months","") \
                                                 .cast("int")/12).cast("int")).withColumnRenamed("loan_term_months","loan_term_years")

In [17]:
loans_term_modfied_df.createOrReplaceTempView("loans")

In [18]:
spark.sql("select distinct(loan_purpose) from loans")

loan_purpose
"guaranteed!"""
and if they are a...
never had any tro...
<br/><br/>Lending...
Bank of America c...
stocks
please feel free ...
I became his prim...
brakes
on one of the bus...


In [19]:
spark.sql("select loan_purpose,count(*) as total from loans group by loan_purpose order by total desc")

loan_purpose,total
debt_consolidation,1277790
credit_card,516926
home_improvement,150440
other,139413
major_purchase,50429
medical,27481
small_business,24659
car,24009
vacation,15525
moving,15402


In [20]:
loan_purpose_lookup =["debt_consolidation","credit_card","home_improvement","other","major_purchase","medical","small_business","car","vacation","moving","house","wedding","renewable_energy","educational"]

In [21]:
loan_purpose_cleaned = loans_term_modfied_df.withColumn(
    "loan_purpose",when(col("loan_purpose") \
    .isin(loan_purpose_lookup),col("loan_purpose")).otherwise("other"))

In [22]:
loan_purpose_cleaned.groupBy("loan_purpose") \
.agg(count('*').alias("total")) \
.orderBy(col("total").desc())

loan_purpose,total
debt_consolidation,1277790
credit_card,516926
home_improvement,150440
other,139667
major_purchase,50429
medical,27481
small_business,24659
car,24009
vacation,15525
moving,15402


In [23]:
loan_purpose_cleaned.count()

2260667

In [24]:
loan_purpose_cleaned.write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", True) \
    .save(f"/user/{username}/lendingclub/cleaned/loans_csv")

In [26]:
loan_purpose_cleaned.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(f"/user/{username}/lendingclub/cleaned/loans_parquet")